# Federation Demo  
In this demo, you will be shown:  
- how to import data from endpoints
- how to see the datasets imported
- how to search for data
- merge data from different tables
- add a table to an existing database

In [ ]:
import csv
import os
from getpass import getpass
from pathlib import Path
from typing import Tuple, List, Dict, Any

import getpass
import paramiko
import shutil
import json
import pandas as pd
from pathlib import Path

from dsi.dsi import DSI
from dsi.dsifederated import DSIFederated
from dsi.sync import Sync
from dsi.utils.federated.federate_datasets import (
    pull_data, 
    just_pull_data, 
    pull_remote_db, 
    get_remote_endpoints,
    pull_data_endpoints
)

from dsi.utils.federation_utils import (
    compute_md5, 
    create_directory, 
    create_hashed_folder_from_path, 
    create_folder,
    combine_csv,
    csv_to_list_of_dicts, 
    deduplicate_keep_latest, 
    get_last_part, 
    human_readable_size, 
    should_download, 
    upsert_records
)

Help for DSI and Federated DSI

In [ ]:
help(DSI)

In [ ]:
help(DSIFederated)

In [ ]:
help(Sync)

## Get Enpoints from HPC

In [ ]:
hpc_name = input("Enter the name of the HPC")
username = input("Enter username: ")
password = getpass.getpass("Enter password: ")

# currently a script setting environment variables but should be load module in the future
script_path='/users/pascalgrosset/dsi_test/load_dsi_endpoints.sh' 

# prefix of the endpoints; environment variables to search for
prefixes=['DSI_ENDPOINT_', 'DIANA_ENDPOINT_'] 

endpoints_location = get_remote_endpoints(hpc_name, username, password, script_path, prefixes)

In [ ]:
endpoints_location

### Federate the data in specified folder

In [ ]:
rel_wrks_folder = "test_federate_07"
workspace_folder = str(Path(rel_wrks_folder).resolve())

database_info = pull_data_endpoints(endpoints_location, hpc_name, workspace_folder)

In [ ]:
database_info

## Instantiate the object

In [ ]:
federated_dbs = DSIFederated(workspace_folder, operating_mode="notebook")

## Browse and search through the data

In [ ]:
federated_dbs.f_list_databases()

## Looking up data

In [ ]:
federated_dbs.f_summary()

In [ ]:
federated_dbs.f_search(query="dens_max")

## Merging data

### Search for databases

In [ ]:
federated_dbs.f_search_for_databases(db="data_subset*")

### Search for the path to a database

In [ ]:
federated_dbs.f_get_db_path(db="data_subset_1.db")

### Load that database

In [ ]:
temp_file = DSI('/Users/pascalgrosset/projects/dsi/dsi_databases_00/87c1361f3c2316dd/data_subset_1.db')

In [ ]:
temp_file.list()

In [ ]:
temp_file.get_table("data", collection=True)

In [ ]:
federated_dbs.f_merge(src_db_id='camouflaged-hare',src_tbl_name='data', dst_db_id='aromatic-dragon',dst_tbl_name='data')

In [ ]:
temp_file.get_table("data", collection=True)

In [ ]:
federated_dbs.f_merge(src_db_id='precious-walrus',src_tbl_name='data', dst_db_id='aromatic-dragon',dst_tbl_name='data')

In [ ]:
temp_file.get_table("data", collection=True)

## Adding another table to the database

In [ ]:
federated_dbs.f_search_for_databases(db="model_subset*")

In [ ]:
federated_dbs.f_add_table(src_db_id='satisfied-whale',src_tbl_name='data', dst_db_id='aromatic-dragon',dst_tbl_name='model')

In [ ]:
temp_file.list()

In [ ]:
temp_file.get_table("model", collection=True)

That database now has datasets which have been pulled from different sites as well as several tables 